# 03 — Évaluation du retrieval et analyse des erreurs

Une méthode de recherche doit retrouver les bonnes sources et les placer à un rang
utile pour l’assistant. Ce notebook confronte les résultats de TF-IDF, BM25 et des
embeddings aux documents attendus pour chaque question.

**Objectif :** comparer la qualité des classements et comprendre les situations dans
lesquelles chaque méthode réussit ou échoue.

L’analyse commence par les métriques pour plusieurs valeurs de K, puis examine les
classements question par question. Les questions hors périmètre sont étudiées
séparément pour distinguer la qualité de recherche de la capacité à refuser.


In [1]:
import logging

# Message macOS concernant une redirection multiprocessus inutilisée ici.
# Les autres avertissements et erreurs restent visibles.
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").addFilter(
    lambda record: record.getMessage() != "NOTE: Redirects are currently not supported in MacOs."
)

In [2]:
from IPython.display import Markdown, display

from contextiq.evaluation.evaluator import RetrievalEvaluator
from contextiq.ingestion.loader import DataLoader
from contextiq.retrieval.bm25 import BM25Retriever
from contextiq.retrieval.embeddings import EmbeddingRetriever
from contextiq.retrieval.tfidf import TfidfRetriever
from contextiq.settings import Settings

settings = Settings.load()
dataset = DataLoader().load()
retrievers = {
    "TF-IDF": TfidfRetriever(dataset.documents),
    "BM25": BM25Retriever(dataset.documents, **settings.retrieval.bm25.model_dump()),
    "Embeddings": EmbeddingRetriever(
        dataset.documents, **settings.retrieval.embeddings.model_dump()
    ),
}

evaluator = RetrievalEvaluator()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 1. Comparer les métriques de classement

- **Precision@K** : nombre de sources pertinentes retrouvées divisé par K, même si moins de K documents sont retournés.
- **Recall@K** : proportion des sources attendues retrouvées.
- **Hit Rate@K** : présence d’au moins une source attendue.
- **MRR@K** : inverse du rang de la première source pertinente, ou zéro si elle est absente du top K.

Les moyennes portent sur les questions avec source. Chacune possède ici une seule
source annotée : Recall@K et Hit Rate@K sont donc identiques. Les questions hors
périmètre sont exclues de ces moyennes, car elles n’ont pas de document attendu.


In [3]:
# Le notebook présente les résultats ; les calculs restent dans RetrievalEvaluator.
reports = {
    (name, k): evaluator.evaluate(engine, dataset.questions, top_k=k)
    for name, engine in retrievers.items()
    for k in settings.evaluation.k_values
}

rows = [
    "| Méthode | K | Questions évaluées | Precision@K | Recall@K | Hit Rate@K | MRR@K |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for (name, k), report in reports.items():
    score = report.summary
    if score is not None:
        rows.append(
            f"| {name} | {k} | {report.in_scope_count} | {score.precision:.3f} | "
            f"{score.recall:.3f} | {score.hit_rate:.3f} | {score.mrr:.3f} |"
        )
display(Markdown("\n".join(rows)))

| Méthode | K | Questions évaluées | Precision@K | Recall@K | Hit Rate@K | MRR@K |
|---|---:|---:|---:|---:|---:|---:|
| TF-IDF | 1 | 5 | 0.600 | 0.600 | 0.600 | 0.600 |
| TF-IDF | 2 | 5 | 0.400 | 0.800 | 0.800 | 0.700 |
| TF-IDF | 3 | 5 | 0.267 | 0.800 | 0.800 | 0.700 |
| BM25 | 1 | 5 | 0.400 | 0.400 | 0.400 | 0.400 |
| BM25 | 2 | 5 | 0.500 | 1.000 | 1.000 | 0.700 |
| BM25 | 3 | 5 | 0.333 | 1.000 | 1.000 | 0.700 |
| Embeddings | 1 | 5 | 0.800 | 0.800 | 0.800 | 0.800 |
| Embeddings | 2 | 5 | 0.500 | 1.000 | 1.000 | 0.900 |
| Embeddings | 3 | 5 | 0.333 | 1.000 | 1.000 | 0.900 |

## 2. Examiner les classements question par question

La comparaison des sources attendues et retrouvées permet de repérer une source
absente, mal classée ou accompagnée de documents non pertinents. Les métriques par
question relient chaque erreur au résultat agrégé.
Les scores bruts des trois méthodes ont des échelles différentes et ne sont pas
comparés directement entre eux.


In [4]:
inspection_k = settings.evaluation.inspection_k

for name in retrievers:
    report = reports[(name, inspection_k)]
    print("\n" + "=" * 70)
    print(f"MÉTHODE : {name} | K = {inspection_k}")
    print("=" * 70)

    # Garder ici les questions avec source ; le hors périmètre est traité après.
    questions = [item for item in report.questions if item.metrics is not None]

    for number, item in enumerate(questions, start=1):
        print(f"\nQuestion {number} — {item.question.q}")
        print("-" * 70)
        print("Sources attendues  :", ", ".join(item.question.gold_docs))
        print("Sources retrouvées :", ", ".join(r.document.id for r in item.results) or "Aucune")
        print()
        print(
            f"  Precision@{inspection_k} : {item.metrics.precision:.3f} | Recall@{inspection_k}   : {item.metrics.recall:.3f}"
        )
        print(
            f"  Hit Rate@{inspection_k}  : {item.metrics.hit_rate:.3f} | MRR@{inspection_k}      : {item.metrics.mrr:.3f}"
        )
        print("-" * 70)


MÉTHODE : TF-IDF | K = 3

Question 1 — Combien de temps de préavis pour un contrat mensuel ?
----------------------------------------------------------------------
Sources attendues  : D1
Sources retrouvées : D1, D4, D3

  Precision@3 : 0.333 | Recall@3   : 1.000
  Hit Rate@3  : 1.000 | MRR@3      : 1.000
----------------------------------------------------------------------

Question 2 — Quels frais si je quitte mon abonnement annuel avant la fin ?
----------------------------------------------------------------------
Sources attendues  : D2
Sources retrouvées : D2, D4, D3

  Precision@3 : 0.333 | Recall@3   : 1.000
  Hit Rate@3  : 1.000 | MRR@3      : 1.000
----------------------------------------------------------------------

Question 3 — Je déménage, puis-je partir sans payer ?
----------------------------------------------------------------------
Sources attendues  : D3
Sources retrouvées : D3, D6, D5

  Precision@3 : 0.333 | Recall@3   : 1.000
  Hit Rate@3  : 1.000 | MRR@3     

## 3. Analyser les résultats hors périmètre

Une question sans réponse attendue peut obtenir des scores positifs avec les trois
méthodes. L’inspection des sources retrouvées permet d’observer ce comportement,
mais ne suffit pas à définir un seuil fiable de refus avec un seul exemple.


In [5]:
for method_number, name in enumerate(retrievers, start=1):
    print("=" * 80)
    print(f"{method_number}. MÉTHODE : {name}")
    print("=" * 80)

    question_number = 0
    for item in reports[(name, settings.evaluation.inspection_k)].questions:
        if item.metrics is not None:
            continue

        question_number += 1
        print(f"\n{method_number}.{question_number}. QUESTION HORS PÉRIMÈTRE")
        print(item.question.q)
        print("-" * 80)
        print("Sources retrouvées :")

        if not item.results:
            print("  Aucun résultat.")
        for rank, result in enumerate(item.results, start=1):
            print(f"  {rank}. Source : {result.document.id} | Score : {result.score:.3f}")

        print("-" * 80)
    print()

1. MÉTHODE : TF-IDF

1.1. QUESTION HORS PÉRIMÈTRE
Quel est le cours de l'action de l'entreprise ?
--------------------------------------------------------------------------------
Sources retrouvées :
  1. Source : D4 | Score : 0.452
  2. Source : D5 | Score : 0.311
  3. Source : D3 | Score : 0.298
--------------------------------------------------------------------------------

2. MÉTHODE : BM25

2.1. QUESTION HORS PÉRIMÈTRE
Quel est le cours de l'action de l'entreprise ?
--------------------------------------------------------------------------------
Sources retrouvées :
  1. Source : D1 | Score : 2.082
  2. Source : D4 | Score : 1.770
  3. Source : D3 | Score : 1.704
--------------------------------------------------------------------------------

3. MÉTHODE : Embeddings

3.1. QUESTION HORS PÉRIMÈTRE
Quel est le cours de l'action de l'entreprise ?
--------------------------------------------------------------------------------
Sources retrouvées :
  1. Source : D2 | Score : 0.360
  2

## Portée de la comparaison

Avec cinq questions pertinentes, les résultats sont descriptifs et les écarts restent
fragiles. D6 n’a pas de question dédiée. Les annotations peuvent également être
incomplètes : une source non annotée est considérée non pertinente dans les calculs.

Un choix de méthode, de K ou de paramètres fondé sur ces questions devra être confirmé
sur un jeu indépendant. Les métriques de retrieval mesurent la qualité des sources
retrouvées ; elles ne mesurent pas encore la fidélité d’une réponse générée.


## Prolongement de l’analyse

Le [notebook sur l’abstention](04_abstention_analysis.ipynb) étudie les décisions de
refus et le compromis entre questions pertinentes refusées et hors périmètre acceptés.
